# Aula 3 - Positional Encoding

In [1]:
import torch
import torch.nn as nn

max_sequence_lenght = 10
d_model = 6

$$PE(pos,2i)=\sin(\frac{pos}{10000^{\frac{2i}{d_\text{model}}}})$$
$$PE(pos,2i+1)=\cos(\frac{pos}{10000^{\frac{2i}{d_\text{model}}}})$$

Podemos reescrever como:

$$PE(pos,i)=\sin(\frac{pos}{10000^{\frac{i}{d_\text{model}}}})\text{ quando i = par}$$
$$PE(pos,i)=\cos(\frac{pos}{10000^{\frac{i-1}{d_\text{model}}}})\text{ quando i = ímpar}$$

In [2]:
even_i = torch.arange(0, d_model, 2).float()
even_i

tensor([0., 2., 4.])

In [3]:
even_denominator = torch.pow(10000, even_i/d_model)
even_denominator

tensor([  1.0000,  21.5443, 464.1590])

In [4]:
odd_i = torch.arange(1, d_model, 2).float()
odd_i

tensor([1., 3., 5.])

In [7]:
odd_denominator = torch.pow(10000, (odd_i-1)/d_model)
odd_denominator

tensor([  1.0000,  21.5443, 464.1590])

Perceba que even_denominator = odd_denominator.

In [12]:
denominator = even_denominator

In [10]:
pos = torch.arange(max_sequence_lenght, dtype=torch.float).reshape(max_sequence_lenght, 1)
pos

tensor([[0.],
        [1.],
        [2.],
        [3.],
        [4.],
        [5.],
        [6.],
        [7.],
        [8.],
        [9.]])

In [13]:
even_PE = torch.sin(pos / denominator)
odd_PE = torch.cos(pos / denominator)

In [14]:
even_PE

tensor([[ 0.0000,  0.0000,  0.0000],
        [ 0.8415,  0.0464,  0.0022],
        [ 0.9093,  0.0927,  0.0043],
        [ 0.1411,  0.1388,  0.0065],
        [-0.7568,  0.1846,  0.0086],
        [-0.9589,  0.2300,  0.0108],
        [-0.2794,  0.2749,  0.0129],
        [ 0.6570,  0.3192,  0.0151],
        [ 0.9894,  0.3629,  0.0172],
        [ 0.4121,  0.4057,  0.0194]])

In [15]:
odd_PE

tensor([[ 1.0000,  1.0000,  1.0000],
        [ 0.5403,  0.9989,  1.0000],
        [-0.4161,  0.9957,  1.0000],
        [-0.9900,  0.9903,  1.0000],
        [-0.6536,  0.9828,  1.0000],
        [ 0.2837,  0.9732,  0.9999],
        [ 0.9602,  0.9615,  0.9999],
        [ 0.7539,  0.9477,  0.9999],
        [-0.1455,  0.9318,  0.9999],
        [-0.9111,  0.9140,  0.9998]])

- Coluna 0: $sin(pos_i/denominator[0])$
- Coluna 1: $sin(pos_i/denominator[1])$
- Coluna 2: $sin(pos_i/denominator[2])$

Não existe divisão de matrizes na álgebra linear formal. O PyTorch faz uma divisão elemento a elemento, usando uma regra chamada Broadcasting.

1. O tensor $A(10\times 1)$ é duplicado 3 vezes na dimensão horizontal

$$\begin{bmatrix} 0 \\ 1 \\ \vdots \\ 9 \end{bmatrix} \longrightarrow \begin{bmatrix} 0 & 0 & 0 \\ 1 & 1 & 1 \\ \vdots & \vdots & \vdots \\ 9 & 9 & 9 \end{bmatrix}$$

2. O tensor $B(1\times 3)$ é duplicado 10 vezes na dimensão vertical

$$\begin{bmatrix} b_0 & b_1 & b_2 \end{bmatrix} \longrightarrow \begin{bmatrix} b_0 & b_1 & b_2 \\ b_0 & b_1 & b_2 \\ \vdots & \vdots & \vdots \\ b_0 & b_1 & b_2 \end{bmatrix}$$

3. O PyTorch divide cada elmento da matriz A expandida pelo elemento correspondente da matriz B expandida.

$$R_{i,j}=\frac{A_{i,0}}{B_{0,j}}$$

- 1° Índice: even[0]
- 2° Índice: odd[0]
- 3° Índice: even[1]
- 4° Índice: odd[1]

In [18]:
stacked = torch.stack([even_PE, odd_PE], dim=2) # Cria uma nova dimensão (dim=2) e junta os tensores par a par
stacked

tensor([[[ 0.0000,  1.0000],
         [ 0.0000,  1.0000],
         [ 0.0000,  1.0000]],

        [[ 0.8415,  0.5403],
         [ 0.0464,  0.9989],
         [ 0.0022,  1.0000]],

        [[ 0.9093, -0.4161],
         [ 0.0927,  0.9957],
         [ 0.0043,  1.0000]],

        [[ 0.1411, -0.9900],
         [ 0.1388,  0.9903],
         [ 0.0065,  1.0000]],

        [[-0.7568, -0.6536],
         [ 0.1846,  0.9828],
         [ 0.0086,  1.0000]],

        [[-0.9589,  0.2837],
         [ 0.2300,  0.9732],
         [ 0.0108,  0.9999]],

        [[-0.2794,  0.9602],
         [ 0.2749,  0.9615],
         [ 0.0129,  0.9999]],

        [[ 0.6570,  0.7539],
         [ 0.3192,  0.9477],
         [ 0.0151,  0.9999]],

        [[ 0.9894, -0.1455],
         [ 0.3629,  0.9318],
         [ 0.0172,  0.9999]],

        [[ 0.4121, -0.9111],
         [ 0.4057,  0.9140],
         [ 0.0194,  0.9998]]])

In [19]:
stacked.shape

torch.Size([10, 3, 2])

In [20]:
PE = torch.flatten(stacked, start_dim=1, end_dim=2) # Fundir dimensões 1 e 2 em uma única dimensão contínua
PE

tensor([[ 0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.0464,  0.9989,  0.0022,  1.0000],
        [ 0.9093, -0.4161,  0.0927,  0.9957,  0.0043,  1.0000],
        [ 0.1411, -0.9900,  0.1388,  0.9903,  0.0065,  1.0000],
        [-0.7568, -0.6536,  0.1846,  0.9828,  0.0086,  1.0000],
        [-0.9589,  0.2837,  0.2300,  0.9732,  0.0108,  0.9999],
        [-0.2794,  0.9602,  0.2749,  0.9615,  0.0129,  0.9999],
        [ 0.6570,  0.7539,  0.3192,  0.9477,  0.0151,  0.9999],
        [ 0.9894, -0.1455,  0.3629,  0.9318,  0.0172,  0.9999],
        [ 0.4121, -0.9111,  0.4057,  0.9140,  0.0194,  0.9998]])

- `[ 0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000]` -> PE do 1° token
- `[ 0.8415,  0.5403,  0.0464,  0.9989,  0.0022,  1.0000]` -> PE do 2° token
- `[ 0.9093, -0.4161,  0.0927,  0.9957,  0.0043,  1.0000]` -> PE do 3° token
                                

In [24]:
import torch
import torch.nn as nn

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_sequence_lenght):
        super().__init__()
        self.max_sequence_lenght = max_sequence_lenght
        self.d_model = d_model

    def forward(self):
        even_i = torch.arange(0, self.d_model, 2).float()
        denominator = torch.pow(10000, even_i/self.d_model)
        pos = torch.arange(self.max_sequence_lenght).reshape(self.max_sequence_lenght, 1)
        even_PE = torch.sin(pos/denominator)
        odd_PE = torch.cos(pos/denominator)
        stacked = torch.stack([even_PE, odd_PE], dim=2)
        PE = torch.flatten(stacked, start_dim = 1, end_dim=2)
        return PE

In [27]:
pe = PositionalEncoding(d_model=6, max_sequence_lenght=10)
pe.forward()

tensor([[ 0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.0464,  0.9989,  0.0022,  1.0000],
        [ 0.9093, -0.4161,  0.0927,  0.9957,  0.0043,  1.0000],
        [ 0.1411, -0.9900,  0.1388,  0.9903,  0.0065,  1.0000],
        [-0.7568, -0.6536,  0.1846,  0.9828,  0.0086,  1.0000],
        [-0.9589,  0.2837,  0.2300,  0.9732,  0.0108,  0.9999],
        [-0.2794,  0.9602,  0.2749,  0.9615,  0.0129,  0.9999],
        [ 0.6570,  0.7539,  0.3192,  0.9477,  0.0151,  0.9999],
        [ 0.9894, -0.1455,  0.3629,  0.9318,  0.0172,  0.9999],
        [ 0.4121, -0.9111,  0.4057,  0.9140,  0.0194,  0.9998]])